# Урок 2. Как исследовать незнакомую таблицу

**Курс:** pandas от нуля до профессионала · **Модуль 01** · Урок 2 из 4

**Как работать:** внимательно читайте теорию, запускайте демонстрационные ячейки, затем выполняйте задания без копирования примеров. У каждого задания есть отдельная ячейка проверки на показанных исходных данных.

> Для заданий вам понадобятся только знания из этого и предшествующих уроков. Интернет не требуется.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
from tests.notebook_checks import expected_for, check_result, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Разведочный анализ

Получив файл, не начинайте сразу вычислять итоговые показатели. Сначала ответьте на вопросы: сколько строк, какие столбцы, есть ли пропуски, какие типы, похожи ли значения на ожидаемые. Это называется **первичный анализ данных** (часть EDA — exploratory data analysis). Он выявляет проблемы до того, как они повлияют на выводы.

Для больших таблиц печать всего DataFrame неудобна: вывод может быть усечён, а работа замедлится. Используйте безопасный набор методов просмотра.

## 2. Просмотр содержимого

- `df.head(n)` — первые `n` строк (по умолчанию 5).
- `df.tail(n)` — последние `n` строк.
- `df.sample(n, random_state=42)` — случайные строки; фиксированный `random_state` позволяет воспроизводить результат.
- `df.shape` — размер без вывода данных.
- `df.columns` — названия колонок.
- `df.index` — метки строк.

```python
import pandas as pd
sensors = pd.DataFrame({"device": ["A", "B", "C", "D"], "reading": [2.3, 5.1, 3.4, 7.0]})
print(sensors.head(2))
print(sensors.sample(2, random_state=5))
```

Методы `head()` и `sample()` возвращают новые таблицы; сам `sensors` от этого не меняется. При `sample(n=...)` количество не может превышать число строк без `replace=True`.

## 3. Типы и пропуски

`df.info()` печатает обзор структуры, типов и непустых значений. Этот метод **печатает**, а не возвращает DataFrame; нельзя писать `df.info().shape`. `df.dtypes` возвращает Series с типами столбцов.

Для подсчёта отсутствующих значений используется `df.isna().sum()`: сначала `isna()` получает таблицу True/False, затем `sum()` считает True в каждом столбце. Для общего числа пропусков — `df.isna().sum().sum()`.

**Важно:** пустая строка `""` обычно не считается пропуском. Числовое `0` тоже не является пропуском. Данные надо интерпретировать с учётом предметной области.

## 4. Описательные статистики

`df.describe()` вычисляет статистику по числовым колонкам: `count`, `mean`, `std`, `min`, квартили 25%/50%/75%, `max`.

Квартиль 50% — медиана: половина наблюдений ниже, половина выше (с оговорками для повторов). Среднее может значительно отличаться от медианы, если есть выбросы.

`df.describe(include="all")` добавляет категориальные данные, где видны `unique`, `top`, `freq` (не все показатели применимы ко всем типам).

```python
measurements = pd.DataFrame({"city": ["А", "А", "Б"], "value": [10, 12, 30]})
print(measurements.describe())
```

`count()` считает только непустые значения в столбце, а не все строки. Временные ряды и логические признаки иногда имеют отдельные правила статистики.

## 5. Уникальные значения и частоты

`df["column"].unique()` возвращает уникальные значения, `nunique()` — их количество, `value_counts()` — частоты.

```python
statuses = pd.Series(["new", "done", "new", None])
print(statuses.value_counts())  # None не учитывается по умолчанию
print(statuses.value_counts(dropna=False))  # включает пропуски
```

Частоты полезны для проверки категорий: например, `Москва`, `москва` и `Москва ` могут считаться разными строками. Очистке категорий посвящён следующий этап курса.

## 6. Алгоритм обследования таблицы

1. Просмотрите `shape` и `columns`.
2. Выведите первые несколько строк.
3. Проверьте `dtypes` и `info()`.
4. Посчитайте пропуски и уникальные значения ключевых категорий.
5. Посмотрите диапазоны чисел.
6. Запишите найденные проблемы, прежде чем изменять данные.

Самопроверка: почему `count()` и количество строк различаются? Почему `sample()` удобнее `head()` для поиска скрытых аномалий?

## Дополнительный разбор: корректность исследования
`head()` показывает только начало, а не весь набор; отсутствие пропусков в первых пяти строках ничего не говорит об остальных. `info()` помогает оценить пропуски через count, а `isna().sum()` считает их напрямую. `describe()` по умолчанию ориентируется на числовые столбцы; для категорий полезны `value_counts()` и `nunique()`.

Различайте `count()` (не-NA значения) и `len()` (все строки). Среднее значение может быть рассчитано по меньшему числу наблюдений, чем вы ожидали. Медиана — значение середины упорядоченного ряда, она меньше реагирует на большие выбросы; среднее учитывает величину каждого наблюдения. При отчётности полезно одновременно показывать количество наблюдений и агрегат.

Прежде чем удалять пропуски, определите их смысл. Пустая текстовая строка — не всегда NA, значение `0` тоже не является NA. Любая необоснованная замена может изменить результат анализа.

---
## Практические задания

**Правила:** выполните ячейку исходных данных, затем пишите обычный код в ячейке решения. Сохраните полученный результат в переменной `result`. Сразу под ней находится **ожидаемый вывод**. Сравните его со своим результатом, затем запустите автопроверку. Функции объявлять не нужно.

**Почему задания отличаются от теории:** в теории показаны простые иллюстрации методов, а здесь другая предметная область и дополнительные условия. Автопроверка сравнивает ваше решение с результатом на приведённых данных.

### Задание 1 · m1_06

Получите первые `n` строк переданного DataFrame. Число `n` может быть разным.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `sku`, `qty`.
- `n` — число строк (`int`); в этом примере: `2`.


**Выполнение:** используйте готовые переменные `df`, `n`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_06: используются переменные df, n

df = pd.DataFrame({
    'sku': ['a', 'b', 'c', 'd'],
    'qty': [4, 6, 8, 10],
})

n = 2

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_06 = expected_for("m1_06", df=df, n=n)


In [ ]:
# m1_06: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, n
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 2 столбца; колонки: `sku`, `qty`.

```text
  sku  qty
0   a    4
1   b    6
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_06", result, _expected_m1_06)

### Задание 2 · m1_07

Получите Series, содержащую число пропусков для каждого столбца таблицы `df`.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `city`, `qty`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_07: используются переменные df

df = pd.DataFrame({
    'city': ['Рязань', None, 'Уфа'],
    'qty': [1.0, 2.0, float("nan")],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_07 = expected_for("m1_07", df=df)


In [ ]:
# m1_07: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 2 значений; имя серии: `None`.

```text
city    1
qty     1
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_07", result, _expected_m1_07)

### Задание 3 · m1_08

Получите словарь с ключами `rows`, `columns`, `names`, где `names` — обычный список имён столбцов.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `code`, `active`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_08: используются переменные df

df = pd.DataFrame({
    'code': [10, 20, 30],
    'active': [True, False, True],
})

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_08 = expected_for("m1_08", df=df)


In [ ]:
# m1_08: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'rows': 3, 'columns': 2, 'names': ['code', 'active']}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_08", result, _expected_m1_08)

### Задание 4 · m1_09

Получите частоты значений столбца `column`, включая пропуски. Порядок элементов оставьте стандартным для pandas.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `status`.
- `column` — название столбца (`str`); в этом примере: `'status'`.


**Выполнение:** используйте готовые переменные `df`, `column`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_09: используются переменные df, column

df = pd.DataFrame({
    'status': ['ok', None, 'ok', 'draft', 'draft'],
})

column = 'status'

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_09 = expected_for("m1_09", df=df, column=column)


In [ ]:
# m1_09: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, column
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.Series`: 3 значений; имя серии: `count`.

```text
status
ok       2
draft    2
None     1
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_09", result, _expected_m1_09)

### Задание 5 · m1_10

Для числового столбца `column` получите словарь с ключами `mean`, `median`, `min`, `max`. Пропуски должны игнорироваться стандартными средствами pandas.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `grade`.
- `column` — название столбца (`str`); в этом примере: `'grade'`.


**Выполнение:** используйте готовые переменные `df`, `column`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m1_10: используются переменные df, column

df = pd.DataFrame({
    'grade': [1.0, 9.0, float("nan"), 7.0],
})

column = 'grade'

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m1_10 = expected_for("m1_10", df=df, column=column)


In [ ]:
# m1_10: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, column
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

Словарь (`dict`) с указанными ключами и значениями.

```python
{'mean': np.float64(5.666666666666667),
 'median': np.float64(7.0),
 'min': np.float64(1.0),
 'max': np.float64(9.0)}
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m1_10", result, _expected_m1_10)

---
## Рефлексия

1. Какие методы и свойства вы сегодня применяли?
2. Какие ошибки возникали и почему?
3. Что изменится, если входная таблица содержит больше строк или иной порядок значений?

Переходите к следующему уроку, когда все задания текущего прошли проверку.